# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [19]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from anthropic import Anthropic

In [20]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()
claude = Anthropic()  # Claude 클라이언트 생성

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [3]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [4]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [14]:
#추가
def make_booking(destination_city):
    print(f"Tool make_booking called for {destination_city}")
    with open("bookings.txt", "a") as f:
        f.write(f"Booking confirmed for {destination_city}\n")
    return f"Booking confirmed for {destination_city}"


In [5]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [15]:
booking_function = {
    "name": "make_booking",
    "description": "Make a booking for the destination city. Use when the user asks to book or reserve a ticket.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the user wants to book a ticket to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


In [16]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": booking_function}
]


In [17]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    func_name = tool_call.function.name
    
    if func_name == "get_ticket_price":
        city = arguments.get("destination_city")
        price = get_ticket_price(city)
        result = {"destination_city": city, "price": price}
        
    elif func_name == "make_booking":
        city = arguments.get("destination_city")
        confirmation = make_booking(city)
        result = {"destination_city": city, "confirmation": confirmation}
    
    else:
        result = {"error": "Unknown function"}
    
    response = {
        "role": "tool",
        "content": json.dumps(result),
        "tool_call_id": tool_call.id
    }
    return response, arguments.get("destination_city")


In [8]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [9]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [11]:
import base64
from io import BytesIO
from PIL import Image
from IPython.display import Audio, display

def talker(message):
    response = openai.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=message)

    audio_stream = BytesIO(response.content)
    output_filename = "output_audio.mp3"
    with open(output_filename, "wb") as f:
        f.write(audio_stream.read())

    # Play the generated audio
    display(Audio(output_filename, autoplay=True))

talker("지언아 보고싶어")

In [21]:
#추가
def translate_with_claude(text, target_lang="Korean"):
    """Claude를 이용해 간단히 번역"""
    prompt = f"Translate the following text into {target_lang}:\n\n{text}"
    response = claude.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

In [28]:
# 언어 코드 매핑 (원하면 필요한 것만 남겨도 됨)
LANG2CODE = {
    "Auto": None,
    "English": "en",
    "Korean": "ko",
    "Japanese": "ja",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Chinese": "zh"
}

def transcribe_audio(audio_path, lang_choice="Auto"):
    """녹음 파일을 Whisper로 '그 언어 그대로' 텍스트화 (번역 아님)"""
    if not audio_path:
        return ""
    try:
        language_code = LANG2CODE.get(lang_choice, None)  # None이면 자동감지
        with open(audio_path, "rb") as f:
            tr = openai.audio.transcriptions.create(
                model="whisper-1",
                file=f,
                # 언어를 지정하면 번역 대신 '그 언어'로 전사됨
                language=language_code  # 예: "en" 고정 시 영어 그대로 나옴
            )
        return tr.text.strip()
    except Exception as e:
        return f"[STT 오류] {e}"


In [30]:
def transcribe_and_clear(audio_path, lang_choice):
    """전사 결과를 텍스트로, 오디오는 즉시 None으로 비워서 재녹음 가능하게"""
    text = transcribe_audio(audio_path, lang_choice)  # 이미 만들어둔 STT 함수
    return text, None  # (entry 값, audio_input 값)


In [31]:
def do_entry(message, history):
    if not message or not message.strip():
        return "", history  # 빈 입력이면 무시
    history += [{"role":"user", "content":message}]
    return "", history


In [23]:
def chat(history):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    image = None
    
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        image = artist(city)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # Comment out or delete the next line if you'd rather skip Audio for now..
    talker(reply)
    # Claude 번역 추가
    translated = translate_with_claude(reply, "Korean")
    print(f"[번역 결과]\n{translated}")

    
    return history, image, translated

In [32]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        with gr.Column():
            translation_output = gr.Textbox(label="Korean Translation", interactive=False)
            image_output = gr.Image(height=400)

    # ── STT 언어 선택 + 오디오 입력 + 재녹음 버튼 ─────────────────────────
    with gr.Row():
        stt_lang = gr.Dropdown(
            choices=["Auto", "English", "Korean", "Japanese", "French", "Spanish", "German", "Chinese"],
            value="English",  # 영어 전사 고정 추천
            label="STT Language"
        )
        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="🎤 Speak (auto-transcribe on stop)"
        )
        record_again = gr.Button("🎙️ Record Again")  # 눌러서 즉시 오디오 리셋
    # ──────────────────────────────────────────────────────────────────

    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:")
    with gr.Row():
        clear = gr.Button("Clear")

    # 녹음 종료 시: 전사 → entry 채움 + 오디오 비움 → 전송 → 답변/번역/이미지
    audio_input.change(
        fn=transcribe_and_clear,
        inputs=[audio_input, stt_lang],
        outputs=[entry, audio_input]          # entry는 텍스트, audio_input은 None으로 초기화
    ).then(
        do_entry,
        inputs=[entry, chatbot],
        outputs=[entry, chatbot]
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, image_output, translation_output]
    )

    # 재녹음 버튼: 오디오 입력만 깔끔히 초기화(체인 없음)
    record_again.click(
        fn=lambda: None,
        inputs=None,
        outputs=audio_input
    )

    # 키보드 입력도 기존처럼 동작
    entry.submit(
        do_entry,
        inputs=[entry, chatbot],
        outputs=[entry, chatbot]
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, image_output, translation_output]
    )

    # 전체 클리어
    def clear_all():
        return None, None, ""
    clear.click(
        clear_all,
        inputs=None,
        outputs=[chatbot, image_output, translation_output],
        queue=False
    )

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\LG\anaconda3\envs\llms\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LG\anaconda3\envs\llms\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LG\anaconda3\envs\llms\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "C:\Users\LG\anaconda3\envs\llms\Lib\site-packages\starlette\applications.py", line 112, in __call__
    await self.middleware_stack(scope, receive, send)
  File "C:\Users\LG\anaconda3\envs\llms\Lib\site-packages\starlette\middleware\errors.py", line 187, in __call__
    raise exc
  File "C:\Users\LG\anac

[번역 결과]
안녕하세요! 오늘 무엇을 도와드릴까요?
Tool get_ticket_price called for Tokyo


[번역 결과]
도쿄행 티켓 가격은 $1400입니다. 예약을 진행하시겠습니까?


[번역 결과]
다음과 같이 한국어로 번역했습니다:

예약과 관련이 없는 요구사항을 표현하신 것 같습니다. 다른 방식으로 여행 계획에 도움을 드릴 수 있는지 말씀해 주시기 바랍니다!
Tool make_booking called for Tokyo


...

[번역 결과]
다음과 같이 한국어로 번역했습니다:

도쿄행 티켓 가격은 1400달러입니다. 예매를 원하시면 알려주세요!
Tool make_booking called for Tokyo


[번역 결과]
東京行 티켓 예약이 완료되었습니다!
Tool get_ticket_price called for 베를린


[번역 결과]
죄송합니다, 하지만 베를린행 티켓 가격을 확인할 수 없습니다. 다른 방면에서 도움이 필요하시다면 말씀해 주세요!


[번역 결과]
마릴린의 목적지 도시를 알려주시기 바랍니다.
Tool get_ticket_price called for Berlin


[번역 결과]
베를린 티켓 가격은 $499입니다.
